# Multifield `solveField_new` test — Stokes system

This notebook validates the new multifield `prepare → solve → reconstruct` pipeline.
The velocity field is full order, while the pressure field uses `reducedOrder=true`,
so the test exercises a block-diagonal transformation with different field spaces.


In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

solve_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "solve.jl"
)

@assert isfile(solve_path) "Copy solve.jl into the LowLevelFEM src directory first: $solve_path"

Base.include(
    LowLevelFEM,
    solve_path
)

using LowLevelFEM: solveField_new
using LowLevelFEM: prepare_multifield_system, solve_linear_system


┌ Warning: Replacing docs for `LowLevelFEM.GlobalField :: Union{}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.SolveRHS :: Union{}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.NodalCoordinateSystem :: Union{}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.Global :: Tuple{Union{ScalarField, TensorField, VectorField}}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.solve_linear_system :: Tuple{Any, Any}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.default_solver :: Tuple{SystemMatrix}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.field_transformation :: Union{Tuple{Problem}, Tuple{Problem, Vector{MultiPointConstraint}}}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253


## Model and block system

The mixed system contains a full-order velocity field `V` and a reduced-order pressure field `P`.
The existing `solveField` result is retained as a reference for this first multifield validation.


In [2]:
structured_rect_mesh(lx=2, order=3)
material = Material("body")

P = Field(
    [material],
    type=:ScalarField,
    fieldName=:p,
    rhsName=:fp,
    dim=2,
    reducedOrder=true
)

V = Field(
    [material],
    type=:VectorField,
    fieldName=:v,
    rhsName=:fv,
    dim=2
)

pres1 = BoundaryCondition("rightbottom", field=P, p=0)
suppT = BoundaryCondition("top", field=V, vx=0, vy=0)
suppB = BoundaryCondition("bottom", field=V, vx=0, vy=0)
support = [pres1, suppT, suppB]

load_v = LoadCondition("body", fvx=1.0, fvy=0.0)
fv = loadVector(V, [load_v])
gp = loadVector(P, [])
F = SystemVector([fv, gp])

μ = 1.0
γ = 1e-1

A = ∫((SymGrad(V) ⋅ SymGrad(V)) * 2μ)
B = ∫(Div(V) ⋅ P)
C = 0 * ∫(P ⋅ P)
D = ∫(Div(V) ⋅ Div(V) * γ)

K = SystemMatrix([
    A + D  B
    B'     C
])


sparse([1, 2, 9, 10, 47, 48, 49, 50, 319, 320  …  3765, 3766, 3775, 3776, 3777, 3778, 3779, 3780, 3781, 3782], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  5673, 5673, 5673, 5673, 5673, 5673, 5673, 5673, 5673, 5673], [1.1558095238095243, 0.5249999999999997, 0.031690476190477324, -0.08312500000000012, -0.3199285714285625, -0.6768750000000011, 0.05742857142856705, 0.2850000000000009, 0.03835119047618928, 0.08312500000000052  …  1.1334656086978145e-16, 0.011571428571428363, 5.079487178094588e-17, -0.03905357142857214, -0.004881696428571539, 0.004881696428571847, 0.03905357142857165, -5.722419066378492e-16, -5.345116710353537e-16, 1.1965255175550027e-15], 5673, 5673)

## Old and new solvers

Both solvers act on the same assembled `SystemMatrix` and `SystemVector`.


In [3]:
v_old, p_old = solveField(
    K,
    F;
    support=support
)

v_new, p_new = solveField_new(
    K,
    F;
    support=support
)

(
    velocity_abs_error = norm(v_new.a - v_old.a),
    velocity_rel_error = norm(v_new.a - v_old.a) / norm(v_old.a),
    pressure_abs_error = norm(p_new.a - p_old.a),
    pressure_rel_error = norm(p_new.a - p_old.a) / norm(p_old.a)
)


LoadError: UndefVarError: `solveField_new` not defined in `Main`
Suggestion: this global was defined as `LowLevelFEM.solveField_new` but not assigned a value.

## Prepared-system check

This verifies the reduced algebra independently of field reconstruction.


In [ ]:
prepared = prepare_multifield_system(
    K,
    F,
    support
)

Xfree = solve_linear_system(
    prepared.A,
    prepared.B;
    solver=:backslash
)

residual = norm(prepared.A * Xfree - prepared.B) / norm(prepared.B)

(
    full_dofs = size(K.A, 1),
    transformed_dofs = size(prepared.T, 2),
    free_dofs = size(prepared.A, 1),
    residual = residual
)


(full_dofs = 5673, transformed_dofs = 4643, free_dofs = 4398, residual = 3.9682376692768e-9)

## Symmetric wrapper

The mixed Stokes matrix is symmetric but indefinite. This deliberately tests the
`:auto_symmetric` path: Cholesky may fail and the solver should then fall back to LU.


In [ ]:
Ks = Symmetric(K)

v_sym, p_sym = solveField_new(
    Ks,
    F;
    support=support
)

(
    velocity_rel_error = norm(v_sym.a - v_new.a) / norm(v_new.a),
    pressure_rel_error = norm(p_sym.a - p_new.a) / norm(p_new.a)
)


[ Info: Symmetric system is not positive definite; using LU factorization.


(velocity_rel_error = 0.0, pressure_rel_error = 0.0)

## Visual check


In [ ]:
showDoFResults(v_new, name="v", visible=true)
showDoFResults(p_new, name="p")


1

In [ ]:
openPostProcessor()


## Multifield + MPC

This section exercises the existing multifield MPC machinery through the new
`prepare → solve → reconstruct` pipeline. Two field-specific constraints are used:

- a velocity MPC tying the right boundary to the `rightbottom` master node;
- a pressure MPC tying `leftbottom` to `rightbottom` while pressure also uses `reducedOrder=true`.

The old and new solvers must produce the same constrained solution.


In [ ]:
mpc_v = MPC(
    master="rightbottom",
    slave="right",
    field=V,
    vx=true,
    vy=true
)

mpc_p = MPC(
    master="rightbottom",
    slave="leftbottom",
    field=P,
    p=true
)

mpcs = [mpc_v, mpc_p]


2-element Vector{MultiPointConstraint}:
 MultiPointConstraint("rightbottom", "right", Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1891, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv, false), Dict{Symbol, Bool}(:vy => 1, :vx => 1))
 MultiPointConstraint("rightbottom", "leftbottom", Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1891, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp, true), Dict{Symbol, Bool}(:p => 1))

In [ ]:
v_old_mpc, p_old_mpc = solveField(
    K,
    F;
    support=support,
    mpc=mpcs
)

v_new_mpc, p_new_mpc = solveField_new(
    K,
    F;
    support=support,
    mpc=mpcs
)

(
    velocity_abs_error = norm(v_new_mpc.a - v_old_mpc.a),
    velocity_rel_error = norm(v_new_mpc.a - v_old_mpc.a) / max(norm(v_old_mpc.a), eps()),
    pressure_abs_error = norm(p_new_mpc.a - p_old_mpc.a),
    pressure_rel_error = norm(p_new_mpc.a - p_old_mpc.a) / max(norm(p_old_mpc.a), eps())
)


(velocity_abs_error = 0.0, velocity_rel_error = 0.0, pressure_abs_error = 0.0, pressure_rel_error = 0.0)

### MPC prepared-system check

Besides comparing reconstructed fields, verify the reduced algebra and show how many
transformed DOFs remain after reduced-order interpolation and MPC elimination.


In [ ]:
prepared_mpc = prepare_multifield_system(
    K,
    F,
    support;
    mpc=mpcs
)

Xfree_mpc = solve_linear_system(
    prepared_mpc.A,
    prepared_mpc.B;
    solver=:backslash
)

residual_mpc = norm(
    prepared_mpc.A * Xfree_mpc - prepared_mpc.B
) / max(norm(prepared_mpc.B), eps())

(
    full_dofs = size(K.A, 1),
    transformed_dofs = size(prepared_mpc.T, 2),
    free_dofs = size(prepared_mpc.A, 1),
    prunable_dofs = length(prepared_mpc.prunable),
    protected_dofs = length(prepared_mpc.protected),
    residual = residual_mpc
)


(full_dofs = 5673, transformed_dofs = 4582, free_dofs = 4339, prunable_dofs = 4582, protected_dofs = 6, residual = 1.7600972558906974e-11)

### Direct MPC relation check

The reconstructed fields should satisfy the imposed equality constraints directly in
the original full nodal space.


In [ ]:
rep_v = LowLevelFEM.mpcRepresentativeMap(V, [mpc_v])
rep_p = LowLevelFEM.mpcRepresentativeMap(P, [mpc_p])

err_v_mpc = maximum(
    abs(v_new_mpc.a[i, 1] - v_new_mpc.a[rep_v[i], 1])
    for i in eachindex(rep_v)
)

err_p_mpc = maximum(
    abs(p_new_mpc.a[i, 1] - p_new_mpc.a[rep_p[i], 1])
    for i in eachindex(rep_p)
)

(
    velocity_mpc_error = err_v_mpc,
    pressure_mpc_error = err_p_mpc
)


(velocity_mpc_error = 0.0, pressure_mpc_error = 0.0)

## Multifield + coordinate system

This test rotates only the velocity field while the pressure field remains scalar and
`reducedOrder=true`. Therefore the global coordinate transformation has the form

$$Q = \operatorname{blockdiag}(Q_v, I_p),$$

while the kinematic/order-reduction transformation has the independent block structure

$$T = \operatorname{blockdiag}(T_v, T_p).$$

The two transformations act on different fields in this example.


In [ ]:
# 90-degree local basis on the complete velocity field:
# local e1 = global +y, local e2 = global -x.
e1_v = VectorField(V, "body", [0.0, 1.0, 0.0])
cs_v = CoordinateSystem(e1_v)

# Reference physical load: global +y.
fv_global_y = loadVector(
    V,
    [LoadCondition("body", fvx=0.0, fvy=1.0)]
)
F_global_y = SystemVector([fv_global_y, gp])

# The same physical load expressed in the rotated local basis:
# local +e1 = global +y.
fv_local_e1 = loadVector(
    V,
    [LoadCondition("body", fvx=1.0, fvy=0.0)]
)
F_local_e1 = SystemVector([fv_local_e1, gp])


SystemVector([0.00015625000000000006; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1891, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv, false), Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1891, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp, true)], [0, 3782])

### Local multifield RHS

An unwrapped `SystemVector` is interpreted in solver coordinates. The reconstructed
velocity is returned in global Cartesian components, so it can be compared directly
with the ordinary global-basis reference solution.


In [ ]:
v_ref_y, p_ref_y = solveField_new(
    K,
    F_global_y;
    support=support
)

v_cs_local, p_cs_local = solveField_new(
    K,
    F_local_e1;
    support=support,
    coordSys=[cs_v]
)

(
    velocity_abs_error = norm(v_cs_local.a - v_ref_y.a),
    velocity_rel_error = norm(v_cs_local.a - v_ref_y.a) / max(norm(v_ref_y.a), eps()),
    pressure_abs_error = norm(p_cs_local.a - p_ref_y.a),
    pressure_rel_error = norm(p_cs_local.a - p_ref_y.a) / max(norm(p_ref_y.a), eps())
)


### `Global(SystemVector)`

The same global Cartesian RHS can also be used directly by wrapping it in `Global(...)`.
The solver then applies $Q^T$ internally before solving.


In [ ]:
v_cs_global, p_cs_global = solveField_new(
    K,
    Global(F_global_y);
    support=support,
    coordSys=[cs_v]
)

(
    velocity_abs_error = norm(v_cs_global.a - v_ref_y.a),
    velocity_rel_error = norm(v_cs_global.a - v_ref_y.a) / max(norm(v_ref_y.a), eps()),
    pressure_abs_error = norm(p_cs_global.a - p_ref_y.a),
    pressure_rel_error = norm(p_cs_global.a - p_ref_y.a) / max(norm(p_ref_y.a), eps())
)


(velocity_abs_error = 1.704395938698347e-12, velocity_rel_error = 5.632425230978137e-12, pressure_abs_error = 1.7803189091892466e-10, pressure_rel_error = 1.4308351965917618e-11)

### Prepared-system check

This also verifies that a velocity coordinate transformation can coexist with pressure
order reduction in the same multifield system.


In [ ]:
prepared_cs = prepare_multifield_system(
    K,
    F_local_e1,
    support;
    coordSys=[cs_v]
)

Xfree_cs = solve_linear_system(
    prepared_cs.A,
    prepared_cs.B;
    solver=:backslash
)

residual_cs = norm(
    prepared_cs.A * Xfree_cs - prepared_cs.B
) / max(norm(prepared_cs.B), eps())

(
    full_dofs = size(K.A, 1),
    coordinate_dofs = size(prepared_cs.Q, 1),
    transformed_dofs = size(prepared_cs.T, 2),
    free_dofs = size(prepared_cs.A, 1),
    pressure_reduced_order = P.reducedOrder,
    residual = residual_cs
)


(full_dofs = 5673, coordinate_dofs = 5673, transformed_dofs = 4643, free_dofs = 4398, pressure_reduced_order = true, residual = 7.437212064045057e-11)

## Coordinate system + MPC

MPC relations are now interpreted in the **local solver components**. With
$x_g = QT x_r$, the MPC is contained in $T$, while $Q$ assigns the physical
global direction of those tied local components.

If master and slave use the same local basis, the result is the ordinary
component-wise equality in global coordinates. If their bases differ, the
global relation automatically contains the relative rotation.


### Same coordinate basis on master and slave

First use the same 90° rotated basis on the complete velocity field. In this case
the local MPC is physically equivalent to the same MPC written in the global basis.
The pressure field remains `reducedOrder=true` and also keeps its own MPC, so this
single test combines multifield, reduced order, two MPCs, and a velocity coordinate
transformation.


In [ ]:
v_ref_qmpc, p_ref_qmpc = solveField_new(
    K,
    F_global_y;
    support=support,
    mpc=mpcs
)

v_qmpc, p_qmpc = solveField_new(
    K,
    F_local_e1;
    support=support,
    mpc=mpcs,
    coordSys=[cs_v]
)

(
    velocity_abs_error = norm(v_qmpc.a - v_ref_qmpc.a),
    velocity_rel_error = norm(v_qmpc.a - v_ref_qmpc.a) / max(norm(v_ref_qmpc.a), eps()),
    pressure_abs_error = norm(p_qmpc.a - p_ref_qmpc.a),
    pressure_rel_error = norm(p_qmpc.a - p_ref_qmpc.a) / max(norm(p_ref_qmpc.a), eps())
)


(velocity_abs_error = 5.022893187962244e-13, velocity_rel_error = 2.323184519138411e-12, pressure_abs_error = 4.566792299105701e-11, pressure_rel_error = 3.602533196079784e-12)

### Single-field regression for `coordSys + MPC`

The same semantics should also work through the single-field preparation path.


In [ ]:
Kv = A + D

v_single_ref = solveField_new(
    Kv,
    fv_global_y;
    support=[suppT, suppB],
    mpc=[mpc_v]
)

v_single_qmpc = solveField_new(
    Kv,
    fv_local_e1;
    support=[suppT, suppB],
    mpc=[mpc_v],
    coordSys=[cs_v]
)

(
    abs_error = norm(v_single_qmpc.a - v_single_ref.a),
    rel_error = norm(v_single_qmpc.a - v_single_ref.a) / max(norm(v_single_ref.a), eps())
)


(abs_error = 6.141220426380141e-15, rel_error = 3.74607354430312e-15)

### Deliberately different master/slave bases

Now let the velocity basis rotate continuously from 0° at the left edge to 30° at
the right edge. The MPC ties the local components of the right boundary to the
`leftbottom` master node. Thus the global relation is

$$u_{g,s} = Q_s Q_m^T u_{g,m}.$$

This is the same algebraic mechanism that can be used for rotated periodic or
sector-symmetry constraints. The example below is only a compact master/slave test;
a true sector model would normally pair corresponding nodes periodically.


In [ ]:
θ = π / 6
Lx = 2.0

function sector_e1(x, y, z)
    α = θ * x / Lx
    return [cos(α), sin(α), 0.0]
end

e1_sector = VectorField(V, "body", sector_e1)
cs_sector = CoordinateSystem(e1_sector)

mpc_sector = MPC(
    master="leftbottom",
    slave="right",
    field=V,
    vx=true,
    vy=true
)

# These supports remove rigid-body modes but leave the master's local y
# component free, so the rotated relation is non-trivial.
support_sector = [
    BoundaryCondition("left", field=V, vx=0),
    BoundaryCondition("lefttop", field=V, vy=0),
    pres1
]


3-element Vector{BoundaryCondition}:
 BoundaryCondition("left", Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1891, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv, false), Dict{Symbol, Union{Function, Number, ScalarField}}(:vx => 0))
 BoundaryCondition("lefttop", Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1891, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv, false), Dict{Symbol, Union{Function, Number, ScalarField}}(:vy => 0))
 BoundaryCondition("rightbottom", Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166

In [ ]:
v_sector, p_sector = solveField_new(
    K,
    Global(F_global_y);
    support=support_sector,
    mpc=[mpc_sector],
    coordSys=[cs_sector]
)

prepared_sector = prepare_multifield_system(
    K,
    Global(F_global_y),
    support_sector;
    mpc=[mpc_sector],
    coordSys=[cs_sector]
)

nV = V.non * V.pdim
Qv = prepared_sector.Q[1:nV, 1:nV]
v_local = Qv' * v_sector.a[:, 1]
rep_sector = LowLevelFEM.mpcRepresentativeMap(V, [mpc_sector])

local_mpc_error = maximum(
    abs(v_local[i] - v_local[rep_sector[i]])
    for i in eachindex(rep_sector)
    if rep_sector[i] != i
)

# Verify the physical rotated relation node by node:
# u_s = Q_s * Q_m' * u_m.
rotation_errors = Float64[]

for node in 1:V.non
    sdofs = [2node - 1, 2node]
    mdofs = [rep_sector[sdofs[1]], rep_sector[sdofs[2]]]

    if mdofs != sdofs
        us = v_sector.a[sdofs, 1]
        um = v_sector.a[mdofs, 1]
        Qs = Matrix(Qv[sdofs, sdofs])
        Qm = Matrix(Qv[mdofs, mdofs])
        push!(rotation_errors, norm(us - Qs * Qm' * um))
    end
end

rotated_global_relation_error = maximum(rotation_errors)

(
    local_mpc_error = local_mpc_error,
    rotated_global_relation_error = rotated_global_relation_error,
    number_of_checked_slave_nodes = length(rotation_errors)
)


(local_mpc_error = 5.640561083533972e-17, rotated_global_relation_error = 0.0, number_of_checked_slave_nodes = 31)

The two errors above should be close to machine precision. The first checks the
actual MPC statement in local coordinates; the second checks its equivalent rotated
global form. No equality of the master and slave coordinate systems is required by
the core—the user is responsible for choosing bases that encode the intended
physical relation.
